In [ ]:
# PyTorch functions/methods helpers

# 6.5.4
class FixedScale(nn.Module):

    def __init__(self, features):
        super().__init__()
        self.register_buffer("scale", torch.ones(features)) # Registers "scale" as a buffer with nn.Module.register_buffer()
                                                            # By default, buffers are persistent=True (included in state_dict()). You can toggle persistent as an argument in register_buffer()

* Custom layers are where you extend PyTorch without leaving PyTorch's model system.

* The goal is to write new computations that still behave like normal modules: inspectable, trainable when appropriate, movable across devices, and saveable through `state_dict`.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain when a new operation deserves to become a module
- write parameter-free and parameterized layers
- register trainable tensors with `nn.Parameter`
- register persistent non-trainable tensor state as a buffer
- prove a custom layer parameter updates after one optimizer step

In [ ]:
import math
from pathlib import Path
import tempfile

import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_scalars(parameters):
    return sum(p.numel() for p in parameters)

# 6.5.0 The Problem This Notebook Solves

Built-in layers cover common operations, but research code and serious engineering often need custom behavior. The mistake is to think custom behavior means abandoning PyTorch conventions.

A good custom layer should still answer the same questions as a built-in layer:

- What computation runs in `forward`?
- Which tensors are trainable parameters?
- Which tensors are persistent but not trainable?
- What input and output shapes does the layer promise?
- Will an optimizer, device move, or save/load call discover the right state?

This notebook builds three categories:

```text
parameter-free layer: computation only (e.g. ReLU)
parameterized layer: computation plus learned tensors (e.g. LayerNorm)
buffer-owning layer: computation plus persistent non-learned tensor state (e.g. BatchNorm, parameterized + buffer owning)
```

* Parameter: "a tensor whose value is learned."
* Buffer: "a tensor whose value is maintained/stored by the model, but isn't learned by the optimizer."

```text
TRAINING                         INFERENCE
────────                         ─────────
batch → mean/variance            individual image
         ↓                              ↓
    normalize                       running mean/variance
         ↓                              ↓
       γ, β                            γ, β
         ↓                              ↓
      output                          output
```

The handoff from earlier Chapter 6 sections is deliberate:
* 6.1 gave modules,
* 6.2 gave parameter ownership,
* 6.3 gave initialization, and
* 6.4 gave shape creation.
* Custom layers combine all of those ideas.

# 6.5.1 Parameter-Free Layers Still Belong in Module Pipelines

Some transformations have no learned weights, but they still change the representation passed to later layers.
* That makes them legitimate model components.

`RowCenter` subtracts each row's own mean.
* If each row is one example, the layer removes that example's average feature level while preserving its relative feature differences.
* The theory-level point is that not every useful model operation is learned.
* Some operations impose a fixed transformation that changes what the learned layers have to handle.

Before running the cell, predict:

- Output shape should match input shape.
- Each row's mean should become zero.
- The layer should expose no trainable parameters.

In [ ]:
class RowCenter(nn.Module):

    def forward(self, X):
        return X - X.mean(dim=1, keepdim=True) # Average across the feature dimension (3), separately for each sample (2) for the shape of (2, 3)
                                               # Mean for X or [[1, 2, 3], [10, 20, 30]] are [[2], [20]]
                                               # Use [1, 2, 3] - [2] and [10, 20, 30] - [20] gives [-1, 0, 1] and [-10, 0, 10]

layer = RowCenter()
X = torch.tensor([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]])
Y = layer(X)

print(Y)
print("row means:", Y.mean(dim=1))
print("parameters:", list(layer.parameters()))

assert shape(Y) == shape(X) # Yes, because shape is unaffected during this arithmetic
assert torch.allclose(Y.mean(dim=1), torch.zeros(2)) # Yes, for Y's shape of (2, 3), the average across the feature dimension (3) gives [[(-1+0+1)/3], [(-10+0+10)/3]], which are [[0], [0]]
assert list(layer.parameters()) == [] # Yes, since there are no learnable parameters registered with PyTorch in RowCenter()

tensor([[ -1.,   0.,   1.],
        [-10.,   0.,  10.]])
row means: tensor([0., 0.])
parameters: []


# 6.5.2 Trainable Custom Layers Use `nn.Parameter`

This custom layer is a small linear layer written by hand. The computation is familiar:

```text
input representation @ weight + bias -> output representation
```

The important part is not the matrix multiply itself. The important part is how the learnable tensors are owned.

Assigning an `nn.Parameter` to `self.weight` or `self.bias` tells PyTorch:

```text
this tensor is learned model state
include it in parameters()
include it in state_dict()
move it when the module moves devices
compute gradients for it during backward
```

That is the difference between a custom layer that merely computes and a custom layer that participates correctly in training.

In [ ]:
class MyLinear(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, X):
        return X @ self.weight + self.bias

layer = MyLinear(3, 2)
X = torch.randn(4, 3)
Y = layer(X)

print("output shape:", shape(Y)) # X shape of (4, 3) @ weight shape of (3, 2) = (4, 2)
print("named parameters:", [(name, shape(p)) for name, p in layer.named_parameters()]) # "weight", "bias" since named_parameters() reports the names of parameters registered by nn.Module, without the "self." prefix

assert shape(Y) == (4, 2)
assert [name for name, _ in layer.named_parameters()] == ["weight", "bias"]

output shape: (4, 2)
named parameters: [('weight', (3, 2)), ('bias', (2,))]


# 6.5.3 Prove the Custom Parameters Actually Update

Never trust a custom trainable layer just because the forward pass runs.

A correct custom layer must satisfy three contracts:

```text
forward produces the expected shape
backward creates gradients for registered parameters
optimizer.step mutates those parameters
```

This cell runs one synthetic update only. It is not trying to learn a meaningful task. It is a wiring test.

If this test fails, a longer training experiment would only hide the bug under more code.

This style should feel familiar from serious integration work, but it stays inside the notebook: tiny proof first, larger use later.

In [ ]:
torch.manual_seed(0)
layer = MyLinear(3, 1)
optimizer = torch.optim.SGD(layer.parameters(), lr=0.1)

X = torch.randn(8, 3)
y = torch.randn(8, 1)

before = layer.weight.detach().clone()
pred = layer(X)
loss = ((pred - y) ** 2).mean()

optimizer.zero_grad()
loss.backward()
optimizer.step()

after = layer.weight.detach().clone()

print("loss:", float(loss.detach()))
print("grad shape:", shape(layer.weight.grad)) # Per 6.5.2, layer.weight was initialized as (in_features, out_features) = (3, 1), layer.weight.grad is also (3, 1)
                                               # In practice, PyTorch's layer modules like nn.Linear(in_features, out_features) store weight as (out_features, in_features) and effectively uses weight.T in mat

print("weight changed:", not torch.allclose(before, after))

assert shape(layer.weight.grad) == shape(layer.weight)
assert not torch.allclose(before, after)

loss: 0.6178309321403503
grad shape: (3, 1)
weight changed: True


# 6.5.4 Buffers Are Persistent State, Not Learned Parameters

**Some tensor state belongs to the module but should not be learned by gradient descent**. That is what buffers are for.

A buffer is appropriate when a tensor should:

- move with the module when calling `.to(device)`
- appear in `state_dict`
- persist as part of the model's behavior
- **not appear in `parameters()`**
- **not be updated by the optimizer**

Running statistics in normalization layers are the classic real example.

The `FixedScale` example is simpler: the scale tensor is persistent state, but it is not something the optimizer should learn.

The theoretical distinction is clean:

```text
parameter: learned state
buffer: persistent non-learned state
ordinary attribute: Python-side state not automatically treated as model tensor state
```

In [ ]:
class FixedScale(nn.Module):

    def __init__(self, features):
        super().__init__()
        self.register_buffer("scale", torch.ones(features))

    def forward(self, X):
        return X * self.scale # input X is multiplied by the factor of scale

layer = FixedScale(3)
print("parameters:", list(layer.named_parameters())) # [] because no nn.Parameter() was registered, instead, "scale" was registered as a buffer with nn.Module.register_buffer() called through self
print("buffers:", [(name, shape(buf)) for name, buf in layer.named_buffers()]) # [("scale", (3,))] because "scale" was registered with register_buffer()
print("state_dict:", list(layer.state_dict().keys())) # ["scale"] because persistent buffers are included in state_dict()
                                                      # Non-persistent buffers are excluded from state_dict(). You can toggle persistent as an argument in register_buffer(), which defaults to True

assert list(layer.named_parameters()) == []
assert list(layer.state_dict().keys()) == ["scale"]

parameters: []
buffers: [('scale', (3,))]
state_dict: ['scale']


# 6.5.5 Break It Deliberately: Forget `nn.Parameter`

This layer looks plausible because the tensor has `requires_grad=True`. But it is still not registered as a module parameter.
* The issue is not whether autograd can theoretically compute a gradient for the tensor.
* The issue is whether PyTorch's model tools can discover that tensor as part of the model's learned state.
* If a tensor has a gradient but is not registered as a parameter via `nn.Parameter()`, `optimizer.step()` will not update it.
* The optimizer asks the module for registered parameters. Since there are none, the optimizer has nothing to update.



This is the same failure mode as 6.2, now inside the custom-layer context.

In [ ]:
class AlmostLinear(nn.Module):

    def __init__(self):
        super().__init__()
        self.weight = torch.randn(3, 1, requires_grad=True)

    def forward(self, X):
        return X @ self.weight

layer = AlmostLinear()
print("parameter list:", list(layer.named_parameters())) # [] since self.weight was not registered with nn.Parameter

try:
    torch.optim.SGD(layer.parameters(), lr=0.1) # Error, since optimizer only uses registered parameters
except ValueError as err:
    print(type(err).__name__)
    print(str(err))
else:
    raise AssertionError("The optimizer should reject an empty parameter list.")

parameter list: []
ValueError
optimizer got an empty parameter list


# 6.5 Checkpoint

Answer these before moving on.

You do not need a separate notes file for chapters; short answers in markdown cells or in your own study notes are enough.

1. What makes a custom layer compatible with the rest of PyTorch's model system?
> A custom layer should inherit from `nn.Module`, call `super().__init__()`, and use `nn.Parameter()` for learnable tensors or `register_buffer()` for persistent non-learned tensor state. Its computation goes in `forward()`

2. Why can `RowCenter` be used in a model even though it has no parameters?
> A layer does not obligate learnable parameters. `RowCenter` performs a valid differentiable tensor operation, so it can transform activations and participate in autograd even though it has no learnable state

3. What does `nn.Parameter` change compared with a plain tensor?
> `nn.Parameter` tells `nn.Module` that the tensor is a learnable parameter. When assigned to a module, it is registered so that `parameters()`, `named_parameters()`, optimizers, and `state_dict()` can discover it

4. How did the optimizer-step drill prove the custom layer is wired correctly?
> The drill showed that the parameter was registered, received a gradient during `backward()`, and then actually changed after `optimizer.step()`. Therefore the parameter was correctly connected to the model/optimizer system

5. What is the difference between a parameter, a buffer, and an ordinary attribute?
> * Params are learnable, meaning their values can be updated by gradient descents/SGD/Adam/other kinds of optmizers to minimize the global loss of a model
> * Buffers are non-learned tensors registered with the module as persistent model state (like `scale` from 6.5.4). It is not updated by the optimizer, but it can be used in computations and can be saved in `state_dict()`
> * Ordinary attribute can be basically any Python object or tensor the module uses internally, including random noise, but PyTorch doesn't automatically treat it as model state

6. When should tensor state be a buffer instead of a parameter?
> Use a buffer when **the tensor is persistent model state that should be saved/moved with the module but should not be learned by the optimizer**